In [280]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

import os
os.chdir("C:/Users/arttu/OneDrive/Tiedostot/Akandi")
#1. Vuodet 2000-2019.
df_raw= pd.read_csv("data/processed/atp_matches_2000_2019_raw.csv")

In [ ]:

df_clean = df_raw.copy()
#Kopioidaan data puhdistusta varten.

#2. Carpet ottelut poistetaan 
df_clean = df_clean[df_clean["surface"].isin(["Hard", "Clay", "Grass"])]
df_clean = df_clean.reset_index(drop=True)
df_clean["surface"].value_counts()

#3. poistetaan ottelun jälkeiset featuret: score minutes, JA kaikki w_ ja l_ prefiksillä alkavat sarakkeet, koska ne kertovat ottelun lopputuloksesta.

drop_cols = (
    [c for c in df_clean.columns if c.startswith(("w_", "l_"))]
    + ["score", "minutes"])

df_clean = df_clean.drop(columns=drop_cols)
#--------------------------------
#4. lisätään vuosi-muuttuja.
df_clean["tourney_date"] = pd.to_datetime(
    df_clean["tourney_date"],
    format="%Y%m%d")

df_clean["year"] = df_clean["tourney_date"].dt.year
df_clean["year"].value_counts().sort_index()

df_clean["tourney_date"].dtype


#korjataan virheellinen pituusarvot (3.0cm)
heights = pd.concat([
    df_clean["winner_ht"],
    df_clean["loser_ht"]])

heights.min(), heights.max()

df_clean = df_clean[
    (df_clean["winner_ht"].between(140, 215)) &
    (df_clean["loser_ht"].between(140, 215))]
#-------------------------------
#8. Seed- ja entry-muuttujat sisältävät runsaasti puuttuvia arvoja ja ovat osittain päällekkäisiä ranking-muuttujien kanssa, minkä vuoksi ne poistetaan.
df_clean[[
    "winner_seed", "winner_entry",
    "loser_seed", "loser_entry"
]].isna().mean() * 100

df_clean = df_clean.drop(columns=[
    "winner_seed", "winner_entry",
    "loser_seed", "loser_entry"])

#Kätisyyskorjaukset
#katsotaan A:n (ambidextrous=molempikätisyys) määrät
pd.DataFrame({
    "winner": df_clean["winner_hand"].value_counts(),
    "loser":  df_clean["loser_hand"].value_counts()}).loc[["A", "U"]]
#A yksi havainto (loser), U:ssa 2 havaintoa (winner) ja 7 havaintoa (loser).
#Poistetaan nämä rivit datasta. Sovittaminen ei mielekästä havaintojen äärimmäisen vähyyden vuoksi.

df_clean = df_clean[
    df_clean["winner_hand"].isin(["R", "L"]) &
    df_clean["loser_hand"].isin(["R", "L"])]

pd.DataFrame({
    "winner": df_clean["winner_hand"].value_counts(),
    "loser":  df_clean["loser_hand"].value_counts()})

,winner,loser
R,49750,49186
L,7034,7598


In [ ]:
#rankingit
#13% winner_rank (ja winner_rank_points) puuttuu 41% ja loser_rank (ja loser_rank_points) puuttuu 13%
df_clean[[
    "winner_rank", "loser_rank",
    "winner_rank_points", "loser_rank_points"
]].isna().mean() * 100

no_rank_players = (
    rank_by_player
    .groupby("player")["rank"]
    .apply(lambda x: x.notna().any())
    .loc[lambda x: x == False])

no_rank_players.index.tolist()
pd.DataFrame({"player": no_rank_players.index})
#41 pelaajaa, joilla ei ole ranking-historiaa lainkaan.



,player
0,Adam Moundir
1,Adolfo Daniel Vallejo
2,Adrian Andreev
3,Akash Wagh
4,Alastair Gray
5,Alex Knaff
6,Americo Venero Montes
7,Brandon Perez
8,Bruno Soares
9,Christian Sigsgaard


In [339]:

pd.crosstab(
    df_clean["winner_rank"].isna(),
    df_clean["winner_rank"].shift(1).isna())
#56 k havaintoa ranking ok. 74:ssä ranking ajanhetkellä t ok, t-1 puuttuu. 73 nyt t puuttuu, t-1 taas ok. 2 puuttuu molemmissa (t, t-1).

df_clean[
    df_clean["winner_rank"].isna() | df_clean["loser_rank"].isna()
][["tourney_date", "winner_name", "loser_name", "winner_rank", "loser_rank"]]

,tourney_date,winner_name,loser_name,winner_rank,loser_rank


In [340]:

#imputoidaan puuttuvat rankingit pelaajan edellisellä tunnetulla rankingillä.
df_clean = df_clean.sort_values("tourney_date")

df_clean["winner_rank"] = (
    df_clean
    .groupby("winner_name")["winner_rank"]
    .ffill())

df_clean["loser_rank"] = (
    df_clean
    .groupby("loser_name")["loser_rank"]
    .ffill())

#aikaisemmat 41 pelaajaa jakautuvat 33 (winner) ja 115 (loser) otteluhavaintoon. Jätetään ne vielä toistaiseksi sellaisenaan, kuten ylhäällä todettiin.
df_clean[["winner_rank", "loser_rank"]].isna().sum()

#tehdään sama rank_points-sarakkeille.
df_clean["winner_rank_points"] = (
    df_clean
    .groupby("winner_name")["winner_rank_points"]
    .ffill())

df_clean["loser_rank_points"] = (
    df_clean
    .groupby("loser_name")["loser_rank_points"]
    .ffill())

#nyt tulee enää pelaajat, joilla ole lainkaan ranking-historiaa. Jätetään nämä sellaisenaan.
df_clean[[
    "winner_rank", "loser_rank",
    "winner_rank_points", "loser_rank_points"
]].isna().sum()

winner_rank           0
loser_rank            0
winner_rank_points    0
loser_rank_points     0
dtype: int64

In [341]:
#Jos jollain pelaajalla ei ole ranking:ia yhdessäkään pelissä niin tällöin voi varmaan olettaa että pelejä ei ole kovinkaan montaa tai että pelaajan taso on ranking sijoilla yms., 
#jolloin voit päättää pelaajalle jonkin ns. alimman mahdollisen rankingin tehdään näin.

#ranking-arvo = pelaajan sijoitus ATP listalla -> 2000. (teoriassa sama kuin viimeinen)
#ranking-pisteet = pelaajan ATP pisteet -> 0

#alin mahdollinen ranking
df_clean[["winner_rank", "loser_rank"]] = (
    df_clean[["winner_rank", "loser_rank"]].fillna(2000))

# alin mahdollinen ranking-pistemäärä
df_clean[["winner_rank_points", "loser_rank_points"]] = (
    df_clean[["winner_rank_points", "loser_rank_points"]].fillna(0))

df_clean[[
    "winner_rank","loser_rank",
    "winner_rank_points","loser_rank_points"]].isna().sum()

winner_rank           0
loser_rank            0
winner_rank_points    0
loser_rank_points     0
dtype: int64

In [ ]:
#datasetissä ei ole nyt yhtään puuttuvia havaintoja.
df_clean.isna().sum()
df_clean.to_csv("data/processed/atp_matches_2000_2019_clean.csv", index=False)